## Nome:Mayara Cristina
## Matricula:574
-----


# **ATV 15 — Testes Categorizados**

Esta atividade pratica os **testes para dados categorizados**: Qui‑quadrado clássico, Qui‑quadrado com correção de Yates, Teste Exato de Fisher e Teste de McNemar.

**Objetivos**
- Montar tabelas de contingência e formular hipóteses (H0/H1).
- Calcular valores **esperados** e a estatística **χ²** (ou **p‑valor**), decidir e concluir.
- Usar `scipy.stats`/`statsmodels` para conferir os cálculos.
- Interpretar resultados nos níveis usuais de significância (1%, 5%, 10%).




## Instruções gerais

- Execute as células na ordem em que aparecem.
- Sempre **declare as hipóteses** antes de calcular (H0 = igualdade; H1 = diferença).
- Para tabelas 2×2, você pode usar:
  - `stats.chi2_contingency(obs, correction=False)` → χ² clássico;
  - `stats.chi2_contingency(obs, correction=True)` → com **Yates**;
  - `stats.fisher_exact(obs, alternative="two-sided")` → **Fisher** (n pequeno ou valores esperados < 5);
  - `mcnemar(tabela_pareada, exact=True/False, correction=True/False)` → **McNemar** (amostras pareadas).

In [1]:
# Imports úteis para toda a atividade
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.precision", 4)



## Funções auxiliares


In [2]:
def esperados_from_obs(obs):
    obs = np.asarray(obs, dtype=float)
    row_sum = obs.sum(axis=1, keepdims=True)
    col_sum = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    return row_sum @ col_sum / total

def decide_from_p(p, alphas=(0.10, 0.05, 0.01)):
    return {a: ("Rejeita H0" if p < a else "Não rejeita H0") for a in alphas}

def decide_from_chi2(chi2, crits):
    return {a: ("Rejeitar H0" if chi2 > crits[a] else "Falha em rejeitar H0") for a in crits}



## Exercício 1 — Qui‑quadrado **clássico** (amostras independentes, **n > 40**)

**Contexto:** Uma empresa de eletrônica deseja avaliar se a ocorrência de **defeitos de soldagem** em placas de circuito impresso (PCIs) está associada ao processo de montagem utilizado. Para isso, foram inspecionadas placas de dois grupos independentes: **Montagem Automatizada** e **Montagem Convencional**.

A tabela apresenta os **valores observados** (2×2), com **n = 160** placas:

| Processo de montagem | **Com defeito** | **Sem defeito** | **Total** |
|----------------------|:---------------:|:--------------:|:---------:|
| **Automatizada**     |        8        |       72       |    80     |
| **Convencional**     |       26        |       54       |    80     |
| **Total**            |       34        |      126       |   160     |

1. Formule H0 e H1 para investigar a associação entre o processo de montagem e a ocorrência de defeitos.
2. Calcule a **tabela de esperados** e a estatística **χ² calculado** (sem correção).
3. Obtenha **gl**, **χ² tabelado** para α∈{10%,5%,1%} (use SciPy) e o **p‑valor**.
4. Decida e conclua se há evidências de associação entre o processo de montagem e a presença de defeitos de soldagem.


In [3]:

# Hipóteses
print("H0: Não existe associação entre o processo de montagem e os defeitos.")
print("H1: Existe associação entre o processo de montagem e os defeitos.")

# Valores observados
obs = np.array([[8, 72],
                [26, 54]])

# Valores esperados
esperados = esperados_from_obs(obs)
print("\nValores esperados:")
print(esperados)

# Qui-quadrado clássico, sem correção
chi2, p, gl, esperado = stats.chi2_contingency(obs, correction=False)

print("\nQui-quadrado calculado:", round(chi2, 4))
print("Graus de liberdade:", gl)
print("P-valor:", round(p, 6))

# Valores críticos
for alfa in [0.10, 0.05, 0.01]:
    critico = stats.chi2.ppf(1 - alfa, gl)
    print(f"\nAlfa = {alfa:.0%}")
    print("Qui-quadrado tabelado:", round(critico, 4))
    print("Decisão:", "Rejeita H0" if p < alfa else "Não rejeita H0")


H0: Não existe associação entre o processo de montagem e os defeitos.
H1: Existe associação entre o processo de montagem e os defeitos.

Valores esperados:
[[17. 63.]
 [17. 63.]]

Qui-quadrado calculado: 12.1008
Graus de liberdade: 1
P-valor: 0.000504

Alfa = 10%
Qui-quadrado tabelado: 2.7055
Decisão: Rejeita H0

Alfa = 5%
Qui-quadrado tabelado: 3.8415
Decisão: Rejeita H0

Alfa = 1%
Qui-quadrado tabelado: 6.6349
Decisão: Rejeita H0


**Conclusão:**
Como o p-valor encontrado foi de 0,000504, sendo menor que os três níveis de significância (1%, 5% e 10%), rejeitamos H0 em todos eles. Isso mostra que existe uma relação entre o tipo de montagem e a quantidade de defeitos. Além disso, podemos perceber que a montagem automatizada teve menos defeitos que a convencional.

## Exercício 2 — Qui‑quadrado **com correção de Yates** (amostras independentes, **20 < n < 40**)

**Contexto:** Durante um experimento com redes de sensores, uma equipe de engenharia comparou dois **protocolos de comunicação**, A e B. Em cada grupo, 17 dispositivos independentes realizaram um teste de envio de dados, sendo classificados conforme o resultado da transmissão.

A tabela apresenta os **valores observados** (2×2), com **n = 34** dispositivos:

| Protocolo | **Transmissão bem-sucedida** | **Falha na transmissão** | **Total** |
|-----------|:---------------------------:|:------------------------:|:---------:|
| **A**     |             13              |            4             |    17     |
| **B**     |              7              |           10             |    17     |
| **Total** |             20              |           14             |    34     |

1. Formule H0 e H1 para verificar se o resultado da transmissão está associado ao protocolo utilizado.
2. Calcule os **esperados** e a estatística **χ² com correção de Yates**.
3. Informe **gl**, **p‑valor** e **decisão** para α∈{10%,5%}, interpretando os resultados nos dois níveis de significância.


In [4]:

# Hipóteses
print("H0: O resultado da transmissão não está associado ao protocolo.")
print("H1: O resultado da transmissão está associado ao protocolo.")

# Valores observados
obs = np.array([[13, 4],
                [7, 10]])

# Valores esperados
esperados = esperados_from_obs(obs)
print("\nValores esperados:")
print(esperados)

# Qui-quadrado com correção de Yates
chi2, p, gl, esperado = stats.chi2_contingency(obs, correction=True)

print("\nQui-quadrado com Yates:", round(chi2, 4))
print("Graus de liberdade:", gl)
print("P-valor:", round(p, 6))

# Decisão para 10% e 5%
for alfa in [0.10, 0.05]:
    print(f"\nAlfa = {alfa:.0%}")
    print("Decisão:", "Rejeita H0" if p < alfa else "Não rejeita H0")


H0: O resultado da transmissão não está associado ao protocolo.
H1: O resultado da transmissão está associado ao protocolo.

Valores esperados:
[[10.  7.]
 [10.  7.]]

Qui-quadrado com Yates: 3.0357
Graus de liberdade: 1
P-valor: 0.081451

Alfa = 10%
Decisão: Rejeita H0

Alfa = 5%
Decisão: Não rejeita H0




**Conclusão:**
O p-valor encontrado foi de 0,081451. Como ele é menor que 0,10, rejeitamos H0 para o nível de 10%. Porém, para 5%, não rejeitamos H0, já que o p-valor é maior que 0,05. Assim, podemos dizer que existe uma associação entre o protocolo e o resultado da transmissão considerando 10% de significância, mas não temos evidências suficientes para afirmar isso com 5%.

## Exercício 3 — **Teste Exato de Fisher** (amostras independentes, **n < 20** ou esperados < 5)

**Contexto:** Uma equipe desenvolveu sensores para monitoramento ambiental e comparou duas técnicas de **vedação de invólucros eletrônicos**: resina epóxi e silicone. Após um ensaio padronizado de exposição à água, cada protótipo foi classificado como **sem infiltração** ou **com infiltração**.

A tabela apresenta os resultados de **n = 16** protótipos independentes:

| Técnica de vedação | **Sem infiltração** | **Com infiltração** | **Total** |
|-------------------|:------------------:|:------------------:|:---------:|
| **Resina epóxi**  |         7          |         1          |     8     |
| **Silicone**      |         2          |         6          |     8     |
| **Total**         |         9          |         7          |    16     |

1. Formule H0 e H1 para investigar a associação entre a técnica de vedação e a ocorrência de infiltração (**duas caudas**).
2. Aplique o **Teste Exato de Fisher** (`alternative="two-sided"`) e obtenha o **p‑valor**.
3. Decida em α=5% e conclua sobre os resultados do ensaio.


In [5]:

# Hipóteses
print("H0: Não existe associação entre a técnica de vedação e a infiltração.")
print("H1: Existe associação entre a técnica de vedação e a infiltração.")

# Valores observados
obs = np.array([[7, 1],
                [2, 6]])

# Teste Exato de Fisher (duas caudas)
odds_ratio, p = stats.fisher_exact(obs, alternative="two-sided")

print("\nOdds Ratio:", round(odds_ratio, 4))
print("P-valor:", round(p, 6))

# Decisão para alfa = 5%
alfa = 0.05

if p < alfa:
    print("Decisão: Rejeita H0")
else:
    print("Decisão: Não rejeita H0")


H0: Não existe associação entre a técnica de vedação e a infiltração.
H1: Existe associação entre a técnica de vedação e a infiltração.

Odds Ratio: 21.0
P-valor: 0.040559
Decisão: Rejeita H0



**Conclusão:**
O p-valor obtido foi de 0,040559, que é menor que 0,05. Por isso, rejeitamos H0. Com esse resultado, podemos concluir que o tipo de vedação tem relação com a ocorrência de infiltração. Também podemos observar que a resina epóxi apresentou menos casos de infiltração que o silicone, mostrando um resultado melhor nesse teste.

## Exercício 4 — **Teste de McNemar** (amostras pareadas: antes × depois)

**Contexto:** Uma equipe de sistemas embarcados avaliou o efeito de uma **atualização de firmware** sobre a comunicação de **44 dispositivos IoT**. Os mesmos dispositivos foram submetidos a um teste de comunicação **antes** e **depois** da atualização, sendo classificados como **Estável** ou **Instável** em cada momento.

A tabela apresenta os **dados pareados** (linhas = Antes; colunas = Depois):

| **Antes / Depois** | **Estável** | **Instável** | **Total** |
|-------------------|:----------:|:-----------:|:---------:|
| **Estável**       |     18     |      3      |    21     |
| **Instável**      |     13     |     10      |    23     |
| **Total**         |     31     |     13      |    44     |

1. Formule H0 e H1 (**duas caudas**) para avaliar se a proporção de dispositivos com comunicação estável mudou após a atualização.
2. Calcule o **Teste de McNemar** com e sem correção de continuidade de Yates; reporte a **estatística** e o **p‑valor** de cada versão.
3. Decida para α=1% e α=5% e conclua sobre o efeito da atualização na estabilidade da comunicação.


In [6]:

# Hipóteses
print("H0: A proporção de dispositivos estáveis não mudou após a atualização.")
print("H1: A proporção de dispositivos estáveis mudou após a atualização.")

# Dados pareados: Antes x Depois
obs = np.array([[18, 3],
                [13, 10]])

# McNemar sem correção
resultado_sem = mcnemar(obs, exact=False, correction=False)

# McNemar com correção de Yates
resultado_com = mcnemar(obs, exact=False, correction=True)

print("\nSem correção:")
print("Estatística:", round(resultado_sem.statistic, 4))
print("P-valor:", round(resultado_sem.pvalue, 6))

print("\nCom correção de Yates:")
print("Estatística:", round(resultado_com.statistic, 4))
print("P-valor:", round(resultado_com.pvalue, 6))

# Decisões para 1% e 5%
for alfa in [0.01, 0.05]:
    print(f"\nAlfa = {alfa:.0%}")

    print("Sem correção:",
          "Rejeita H0" if resultado_sem.pvalue < alfa else "Não rejeita H0")

    print("Com correção:",
          "Rejeita H0" if resultado_com.pvalue < alfa else "Não rejeita H0")


H0: A proporção de dispositivos estáveis não mudou após a atualização.
H1: A proporção de dispositivos estáveis mudou após a atualização.

Sem correção:
Estatística: 6.25
P-valor: 0.012419

Com correção de Yates:
Estatística: 5.0625
P-valor: 0.024449

Alfa = 1%
Sem correção: Não rejeita H0
Com correção: Não rejeita H0

Alfa = 5%
Sem correção: Rejeita H0
Com correção: Rejeita H0


**Conclusão:**
No teste sem correção, encontramos um p-valor de 0,012419 e, com a correção de Yates, o valor foi de 0,024449. Nos dois casos, rejeitamos H0 para 5%, mas não para 1%. Isso indica que a atualização do firmware realmente teve efeito na estabilidade dos dispositivos, considerando o nível de 5%. Como mais dispositivos passaram de instáveis para estáveis do que o contrário, podemos perceber que a atualização trouxe uma melhora na comunicação.